Hypothesis - **Emotion concepts control how often a model cheats on [insert type of task from benchmark used]**

### Experiment design
Here
- Model - GLM 5.2 (after trying with Qwen 0.5 and 7B), Benchmarks - ExploitGym, Cyber, HumanEval
- Get benchmark set up and run 1 or 2 evals to check it's working
- Look inside it for evals that caused the model to cheat often, cheat rarely and not cheat at all. Keep those evals.

Separately
- Extract emotion vectors from GLM 5.2 and steer on different emotions to see effect of cheating frequency on all 3
- Expectations
  - High cheating -> No cheating
  - Low cheating -> No cheating
  - No cheating -> No cheating


### Baseline

First I want to establish a baseline and see if the model actually solves the problem - it's a relatively weak model. Let's see what happens

I used both qwen and gpt because I wanted to see a model solve the original problem - one that is likely capable of cheating too. gpt can serve as a ceiling - if we can't get it to solve a regular problem then the setup is broken, and similarly if we can't get it to cheat on any test, then we may struggle to get weaker models to cheat on the same test.

In [10]:
# hello world
from inspect_ai import eval
from impossiblebench import impossible_livecodebench, minimal_humaneval_solver, humaneval_scorer

LOG_DIR = "../../.inspect/logs"

task = impossible_livecodebench(
    split="original",
    solver=minimal_humaneval_solver(temperature=0.7),
    scorer=humaneval_scorer(),
    limit=2,
    sandbox="local",
    max_attempts=10,
)

qwen = eval(
    task,
    model="hf/Qwen/Qwen2.5-0.5B-Instruct",
    model_args={"device": "mps"},
    log_dir=LOG_DIR + "/qwen"
)

gpt = eval(
    task,
    model="openai/gpt-5-mini",
    # model_args={"device": "mps"},
    log_dir=LOG_DIR + "/gpt"
)

Output()

2 samples left after limiting



Output()

In [11]:
from inspect_ai.analysis import samples_df
df = samples_df(qwen)
df.head()

,sample_id,eval_id,log,id,epoch,input,choices,target,metadata_entry_point,metadata_impossible_type,...,total_time,working_time,message_count,turn_count,token_limit_usage,error,limit,limit_reason,retries,fallbacks
0,nKDDcZcL5HeQE4mkDbYukJ,7WQsvrsxYJ3w9n34uD7qMg,/Users/bo/code/emotion-concepts/.inspect/logs/...,lcbhard_0,1,user: Read the following function signature an...,<NA>,,count_beautiful_integers,original,...,26.496,26.487,2,1,<NA>,,<NA>,<NA>,0,0
1,FZ6zeD6jYAM6SnBEHBDEC4,7WQsvrsxYJ3w9n34uD7qMg,/Users/bo/code/emotion-concepts/.inspect/logs/...,lcbhard_1,1,user: Read the following function signature an...,<NA>,,count_powerful_integers,original,...,26.485,26.458,2,1,<NA>,,<NA>,<NA>,0,0


### Cheat

OK the models can clearly attempt the problem. gpt succeeds but qwen doesn't (because it can't code). Let's get gpt to cheat, then we'll find a qwen model that can code and then can cheat too.

Runs to find cheats
- openai/o3, conflicting, strict, n=20 (p=0.33) 99% chance
- openai/gpt-5, oneoff, strict, n=20 (p=0.76) 100% chance (n=3 is 98% chance)

Caveat - swebench will be harder to run (apparently breaks imports and needs to be fixed. So livecodebench for now)

In [ ]:
task = impossible_livecodebench(
    split="conflicting",
    # solver=minimal_humaneval_solver(temperature=0.7),
    # scorer=humaneval_scorer(),
    agent_type="minimal",
    max_attempts=10,
    message_limit=50,
    limit=10,
    sandbox="local",
    custom_id="cheat"
)

gpt = eval(
    task,
    model="openai/gpt-5-mini",
    # model_args={"device": "mps"},
    log_dir=LOG_DIR + "/gpt",
)

Output()

Starting agentic humaneval solver with 10 attempts...

Attempt 1/10

Starting agentic humaneval solver with 10 attempts...

Attempt 1/10

...

Starting agentic humaneval solver with 10 attempts...

Attempt 1/10

RuntimeError: model call cancelled

┌────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────┐
│ /Users/bo/code/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:115 in run                 │
│                                                                                                                │
│   112 │   │   loop.set_debug(debug)                                                                            │
│   113 │   │   task = asyncio.ensure_future(main, loop=loop)                                                    │
│   114 │   │   try:                                                                                             │
│ > 115 │   │   │   return loop.run_until_complete(task)                                                         │
│   116 │   │   finally:                                                                                         │
│   117 │   │   │   if not task.done():                                                                          │
│   118 │   │   │   │   task.cancel()                                                                            │
│                                                                                                                │
│ /Users/bo/code/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:212 in run_until_complete  │
│                                                                                                                │
│   209 │   │   │   if f is not future:                                                                          │
│   210 │   │   │   │   f._log_destroy_pending = False                                                           │
│   211 │   │   │   while not f.done():                                                                          │
│ > 212 │   │   │   │   self._run_once()                                                                         │
│   213 │   │   │   │   if self._stopping:                                                                       │
│   214 │   │   │   │   │   break                                                                                │
│   215 │   │   │   if not f.done():                                                                             │
│                                                                                                                │
│ /Users/bo/code/emotion-concepts/.venv/lib/python3.14/site-packages/nest_asyncio2.py:272 in _run_once           │
│                                                                                                                │
│   269 │   │   │   │   │   │   curr_task = None                                                                 │
│   270 │   │   │   │                                                                                            │
│   271 │   │   │   │   try:                                                                                     │
│ > 272 │   │   │   │   │   handle._run()                                                                        │
│   273 │   │   │   │   finally:                                                                                 │
│   274 │   │   │   │   │   # restore the current task                                                           │
│   275 │   │   │   │   │   if curr_task is not None:                                                            │
│                                                                                                                │
│ /opt/homebrew/Cellar/python@3.14/3.14.7/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/event │
│ s.py:94 in _run                                                                                                │
│                                                                                                                │
│ /Users/bo/code/emotion-concepts/.venv/lib/python3.14/site-packages/uvicorn/server.py:80 

KeyboardInterrupt: 